# Automatische Erkennung von Einstellungen mit *TransNetV2*

Dieses Notebook annotiert die Abfolge und die Länge der Einstellungen innerhalb einer oder mehrerer Videodateien mithilfe der **Shot Boundary Detection**, also der automatisierten Erkennung von Übergängen zwischen Einstellungen. Als Ergebnis entstehen:

- eine Tabelle im `.csv`-Format mit Start, Ende, Mittelpunkt und Dauer jeder erkannten Einstellung,
- eine weitere Tabelle im `.csv`-Format zu den Metadaten der Analyse (**vgl. Kapitel XXX 4.5??**),
- eine ELAN-Datei (`.eaf`) zum Import und zur Sichtung der Annotationsdaten in *VIAN* oder anderen Programmen zur Videoannotation (**vgl. Kapitel xxx**).

Der Arbeitsablauf ist in vier Abschnitte gegliedert: Festlegen der Parameter, technische Vorbereitung, Analyse bzw. Anwendung des Erkennungsmodells und Export der Ergebnisse. Die Texte zwischen den Codeblöcken erläutern vor allem den Zweck dieser einzelnen Schritte und die Bedeutung der erzeugten Dateien, während die Kommentare innerhalb des Codes auf Englisch dessen Funktionsweise vermitteln.

Wir empfehlen bei der Bearbeitung dieses Kapitels in Form eines Jupyter Notebooks die Codeblöcke einzeln nacheinander auszuführen und nicht mit Funktionen wie "Alle ausführen" in Colab oder "Run All Cells" in anderen Anwendungen zur Ausführung von Jupyter Notebooks. Die separate Ausführung der Codeblöcke erleichtert das Verständnis der mit ihnen verbundenen Funktionen und gewährleistet, dass alle vorbereitenden Schritte für nachfolgende Codeblöcke bereits durchlaufen wurden.




## Festlegen der Parameter

In diesem Abschnitt werden die zentralen Parameter für die spätere Analyse festgelegt. Sie können an dieser Stelle angepasst werden, ohne Änderungen im weiteren Code vornehmen zu müssen. Zum Erlernen der Funktionen des Codes in diesem Notebook und für einen ersten Probelauf empfiehlt es sich, die voreingestellten Werte beizubehalten.

### Ordnerpfade

Die Variable `VIDEO_PATH` benennt den Ordner, innerhalb dessen die zu analysierenden Videodateien abgelegt werden. Falls dieser noch nicht existieren sollte, wird ein Ordner mit entsprechender Benennung angelegt. Die einzelnen Videos können dabei sowohl in Unterordnern, als auch auf der höchsten Ebene des Ordners abgelegt werden:

#### Option 1: Videodateien direkt im Videoordner

```text
videos/
├── video_01.mp4
├── video_02.mp4
└── video_03.mp4
```

Alle zu analysierenden Videodateien befinden sich direkt auf der höchsten Ebene des durch `VIDEO_PATH` festgelegten Ordners.

#### Option 2: Videodateien in Unterordnern

```text
videos/
├── film_01/
│   └── video_01.mp4
├── film_02/
│   └── video_02.mp4
└── film_03/
    └── video_03.mp4
```

Die zu analysierenden Videodateien befinden sich in einzelnen Unterordnern innerhalb des durch `VIDEO_PATH` festgelegten Ordners.

Eine Kombination beider Ablageoptionen ist ebenfalls möglich.

**Wichtig** ist, dass jede Videodatei einen einzigartigen Namen tragen muss. Für die Analyse dürfen also keine Dateien mit einem identischen Dateinamen in den Ordner abgelegt werden, da dies zu Fehlern bei der Erstellung der Annotationsdaten führen kann.


In diesen Ordner werden auch die für diese *OER* vorbereiteten Testdateien im zip-Format abgelegt. Diese sind für die Ausführung dieses Jupyter Notebooks zu Lernzwecken vorgesehen und wir empfehlen, diese hierfür auch zu verwenden. Ist deren Download nicht erwünscht, kann die Variable `DOWNLOAD_TESTFILES` auf `False` gesetzt werden. In diesem Fall müssen eigene Videodateien in den Ordner "videos" kopiert werden.

Zusätzlich können auch eigene ZIP-Archive von Online-Speicherplätzen heruntergeladen werden. Hierzu muss ein entsprechender Download-Link innerhalb von Anführungszeichen der Variable `CUSTOM_ZIP_LINK` zugeordnet werden, z.B. `CUSTOM_ZIP_LINK = "https://example.org/video_archive.zip"`. Wichtig ist dabei, dass es sich um einen Direktlink zur `.zip`-Datei handelt, nicht um einen Link zu einer Website. Außerdem sollte beim Download mehrerer Archive nacheinander jeweils die Variable `CUSTOM_ARCHIVE_NAME` geändert werden, da Archive mit demselben Namen nicht erneut heruntergeladen werden. Die Option zum Download eigener Dateien ist besonders bei der Analyse von Langfilmen in *Google Colab* hilfreich, da die Videodateien dadurch nicht manuell und unter erheblichem Zeitaufwand in die *Colab*-Umgebung hochgeladen werden müssen. Der Download von einem Online-Speicher wie z.B. einem Cloud-Dienst benötigt in der Regel weniger Zeit.

Die Variable `EXPORT_PATH` bezeichnet den Pfad zum Exportordner.

### Festlegung des Schwellenwerts.

Besonders wichtig ist der Schwellenwert `THRESHOLD`. Zur Bedeutung von Schwellenwerten vgl. auch **Kapitel xx**. Er bestimmt, ab welcher vom Modell berechneten Wahrscheinlichkeit ein Bildwechsel als Einstellungsgrenze gewertet und als Annotation in die Ausgabe der Ergebnisse aufgenommen wird. Ein höherer Wert führt tendenziell zu weniger, aber sichereren Treffern. Ein niedrigerer Wert erkennt mehr mögliche Übergänge, führt allerdings auch zu einer höheren Wahrscheinlichkeit falsch-positiver Ergebnisse ("*false positives*"). Der Werteumfang für die Wahrscheinlichkeit liegt dabei zwischen `0.0` für 0% und `1.0` für 100%.



In [1]:
# Path to the directory containing the video files to be analysed.
# The default relative path is resolved from the location where the notebook is executed.
VIDEO_PATH = "videos"

# The video directory is created automatically if it does not exist yet.
# `exist_ok=True` prevents an error if the directory already exists.
import os
os.makedirs(VIDEO_PATH, exist_ok=True)

# Path to the directory for the export of the analysis results
EXPORT_PATH = "export"
os.makedirs(EXPORT_PATH, exist_ok=True)

# Minimum probability at which TransNetV2 treats a frame as a confirmed shot boundary.
# The value ranges from 0 to 1.
THRESHOLD = 0.5

# If True, a provided test video is downloaded and extracted.
# Set this option to False when analyzing your own files.
DOWNLOAD_TESTFILES = True

# Add your own custom link inside quotes to download additional files into the video folder.
CUSTOM_ZIP_LINK = "insert link here"
# Change the name of the custom archives. A new archive with the same name will not be downloaded. 
CUSTOM_ARCHIVE_NAME = "custom_archive"

# Separator line for visually clearer text output.
DIVIDER = "─" * 100

---
## Technische Vorbereitung

Bevor die Videos analysiert werden können, müssen die benötigten Softwarepakete installiert, in das Notebook eingebunden und das vortrainierte TransNetV2-Modell geladen werden.

### Installation der Programmbibliotheken

Der folgende Codeblock installiert externe Programmbibliotheken, die nicht in der Standardinstallation von *Python* enthalten sind. Dazu zählen neben dem Modell `TransNetV2` zur Erkennung der Einstellungsübergänge
- `pandas` zum Bearbeiten und Exportieren von tabellarischen Daten,
- `opencv-python` und `ffmpeg` zur Videobearbeitung,
- und `pympi-ling` zum Schreiben von *ELAN*-Dateien (`.eaf`).

Für die wichtigsten Bibliotheken wird eine spezifische Version installiert, um deren interdependente Funktionsweise zu garantieren.

In einer lokalen und bei einer erneuten Ausführung des Notebooks identischen Arbeitsumgebung muss diese Zelle nicht bei jeder Sitzung erneut ausgeführt werden. In *Google Colab* ist eine erneute Installation nach dem Neustart der Laufzeit hingegen notwendig, da alle speziell installierten Programmbibliotheken beim Beenden der Laufzeit gelöscht werden.


In [2]:
print("Installing dependencies.")

# "-q" stands for "quiet" and reduces the amount of installation output.

# The pretrained model and its PyTorch implementation.
!pip install -q transnetv2-pytorch==1.0.5

# Table processing and .csv file export.
!pip install -q pandas

# Reading video files and accessing technical video metadata.
!pip install -q opencv-python==5.0.0.93
!pip install -q static-ffmpeg==3.0

# Creating ELAN-annotation files in the .eaf-format.
!pip install -q pympi-ling
print("Dependencies installed.")

Installing dependencies.
Dependencies installed.


### Import der Programmbibliotheken

Durch den Import werden die installierten Funktionen nun in der aktuellen Notebook-Sitzung verfügbar. Die Bibliotheken übernehmen unterschiedliche Aufgaben, die im Codeblock jeweils kurz erläutert werden.


In [3]:
# Pretrained neural network for shot-boundary detection.
from transnetv2_pytorch import TransNetV2

# Running and managing the model
import torch

# Handling file and directory paths
from pathlib import Path

# Processing of numerical array, including video frames and model predictions
import numpy as np

# Editing and Exporting tabular data
import pandas as pd

# Writing ELAN files.
import pympi

# Representing and calculating time values.
import datetime
import time

# Reading videos and their technical properties.
import cv2
import static_ffmpeg

# Downloading a file from a URL and extracting ZIP archives.
from urllib.request import urlretrieve
import zipfile

print("Dependencies imported.")

Dependencies imported.


### Download des Analysematerials
Zum Download des Testmaterials und eines eigenen `.zip`-Archivs wird eine wiederverwendbare Funktion definiert und direkt angewendet. 

Falls bei der [Festlegung der Parameter](#festlegen-der-parameter) `DOWNLOAD_TESTFILES = True` gesetzt ist, lädt das Notebook das `ZIP`-Archiv `shotDetection_testfiles.zip` herunter, das zwei Trailer zu den Konrad Wolf Filmen **XX** und **YY** enthält **[CHANGE THIS]**. Liegt `shotDetection_testfiles.zip` aufgrund einer vorherigen Ausführung bereits auf derselben Ordnerebene wie das Notebook, wird die Datei nicht erneut heruntergeladen. Anschließend wird ihr Inhalt in den zuvor festgelegten Videoordner entpackt. Zu Lernzwecken im Rahmen der OER empfehlen wir den Download und die Arbeit mit den Testdateien im Notebook.

Dieser Schritt ist für die Arbeit mit eigenen Video- und Film-Korpora nicht erforderlich. Eigene Videodateien können direkt im `VIDEO_PATH` Ordner oder in dessen Unterordnern abgelegt werden. In diesem Fall kann `DOWNLOAD_TESTFILES = False` gesetzt werden. Außerdem kann mittels des Parameters `CUSTOM_ZIP_LINK` ein eigenes `.zip`-Archiv heruntergeladen werden, wie bereits eingangs im Kontext der [Festlegung der Parameter](#festlegen-der-parameter) erläutert. In beiden Fällen sollte der Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) bereits ausgeführt worden sein, um falls notwendig den Videoordner erst zu erstellen.

In [4]:
# Define a reusable function for downloading .zip archives.
# This function takes three arguments: The download link, the name for the downloaded archive, and the directory to unzip the archive into.
def download_zip(url: str, archive_name: str, target_directory: str):

    # Add .zip to the filename and turn it into a relative path in the working directory.
    archive_path = Path(f"{archive_name}.zip")

    # Download only if the archive is not already present in the working directory.
    if not archive_path.is_file():
        # Download the file.
        print(f"Downloading {archive_path}...")
        urlretrieve(url, archive_path)
        print(f"{archive_name} downloaded.")
    else:
        print(f"{archive_name} already downloaded.")

    # Unzip the file to the target_directory.
    target_directory = Path(target_directory)
    with zipfile.ZipFile(archive_path,"r") as zip_ref:
        zip_ref.extractall(target_directory)
        print(f"Extracted {archive_name} to {target_directory}.")

# Download and extract the testfiles if the setting is enabled. 
if DOWNLOAD_TESTFILES == True:
    print("Testfile download requested.")
    
    # Download the testfiles
    url = ("https://owncloud.gwdg.de/index.php/s/C45WJA0iKdZ6uqn/download")
    download_zip(url, "shotDetection_testfiles", VIDEO_PATH)
else:
     print("No testfile download requested.")

# Download and extract the custom archive if a link is provided.
if CUSTOM_ZIP_LINK != "insert link here":
    print("Custom file download requested.")
 
    # Download the custom video archive.
    download_zip(CUSTOM_ZIP_LINK, CUSTOM_ARCHIVE_NAME, VIDEO_PATH)

Testfile download requested.
shotDetection_testfiles already downloaded.
Extracted shotDetection_testfiles to videos.



### Vorbereitung des `TransNetV2`-Modells

In diesem Schritt wird das *Machine-Learning-Modell* zur Erkennung der Einstellungsübergänge vorbereitet. ´TransNetV2` ist ein bereits trainiertes neuronales Netz. Es berechnet für jeden Frame eines Videos einen Wahrscheinlichkeitswert, der angibt, wie wahrscheinlich es ist, dass an dieser Stelle ein Übergang zwischen zwei Einstellungen stattfindet.

Mit `model.eval()` wird das neuronale Netz in den Auswertungsmodus versetzt. Dieser Modus ist für die Anwendung eines trainierten Modells vorgesehen, nicht für dessen weiteres Training.

In [5]:
# Create an instance of the pretrained model.
model = TransNetV2()

# Activate evaluation mode: the model should make predictions rather than continue training.
model.eval()

# Display the compute device being used, usually the CPU or, if configured accordingly, the GPU.
print(f"Model loaded on: {next(model.parameters()).device}")

# Add the programs provided by "static-ffmpeg" to the search path.
# TransNetV2 requires ffmpeg/ffprobe to decode video files reliably.
static_ffmpeg.add_paths()

ℹ️  MPS device detected but not used due to numerical inconsistency issues.
   Use --device mps to explicitly enable MPS (faster but inconsistent results).
Model loaded on: cpu


True

---
## Einlesen der Videos und Analyse der Einstellungsübergänge

Hier beginnt die eigentliche Analyse. Zunächst werden die Videodateien erfasst und deren Metadaten gesammelt. Anschließend führt das Notebook mithilfe des Modells die eigentliche Einstellungserkennung aus und bereitet deren Ergebnis tabellarisch auf.

### Erfassung der technischen Videometadaten

Für jede Videodatei werden deren 
- Ablageort (`path`), 
- Auflösung (`frame_height`& `frame_width`), 
- Anzahl an Frames (`frame_count`), 
- Bildrate (`frame_rate`) 
- und Laufzeit (`duration_hh_mm_ss`) 

bestimmt. Diese Angaben sind sowohl für die Interpretation des Analyseergebnisses als auch für die spätere Umrechnung von Framepositionen in Sekunden notwendig.

Die Informationen aller Videos werden im **Dictionary** `videos` gesammelt. Ein **Dictionary** ist eine Python-Datenstruktur, in der jedem Schlüssel (`key`) ein Wert zugeordnet wird. Im vorliegenden Fall dient der Name eines Videos als Schlüssel. Der zugehörige Wert ist ein weiteres Dictionary, das den Dateipfad, technische Metadaten und später auch die Analyseergebnisse enthält.

In [6]:
# Supported video file extensions.
VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv",
    ".flv", ".webm", ".mpeg", ".mpg", ".m4v", ".3gp"
}

# Store each video's path and technical metadata.
# Video filenames must be unique across all subdirectories.
videos = {}

# Recursively search the input directory for video files.
for video_path in Path(VIDEO_PATH).rglob("*"):

    # Ignore directories and other non-file entries.
    if not video_path.is_file():
        continue

    # Ignore files inside Jupyter checkpoint directories.
    if ".ipynb_checkpoints" in video_path.parts:
        continue

    # Ignore unsupported file formats.
    if video_path.suffix.lower() not in VIDEO_EXTENSIONS:
        continue

    # Use the filename without its extension as the dictionary key.
    video_name = video_path.stem

    # Open the video without loading the complete file into memory.
    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():
        print(f"Video could not be opened: {video_path}")
        capture.release()
        continue

    # Read the video's technical metadata.
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_rate = capture.get(cv2.CAP_PROP_FPS)

    # Release the video file after reading its metadata.
    capture.release()

    # Skip files with invalid metadata.
    if frame_rate <= 0 or frame_count <= 0:
        print(f"Video contains invalid metadata: {video_path}")
        continue

    # Calculate the video duration and convert it to HH:MM:SS.
    duration_hh_mm_ss = str(
        datetime.timedelta(seconds=round(frame_count / frame_rate))
    )

    print(f"Video file: {video_name}")
    print(
        f"Resolution: {frame_width}x{frame_height} | "
        f"Frames: {frame_count} | "
        f"Frame Rate: {frame_rate:.3f} | "
        f"Duration: {duration_hh_mm_ss}"
    )
    print(DIVIDER)

    # Store the metadata for the subsequent analysis steps.
    videos[video_name] = {
        "path": video_path,
        "frame_count": frame_count,
        "frame_height": frame_height,
        "frame_width": frame_width,
        "frame_rate": frame_rate,
        "duration_hh_mm_ss": duration_hh_mm_ss
    }

Video file: Der geteilte Himmel - DEFA-Trailer
Resolution: 1920x1080 | Frames: 5225 | Frame Rate: 25.000 | Duration: 0:03:29
────────────────────────────────────────────────────────────────────────────────────────────────────


### Anwendung des Modells

Im folgenden Codeblock werden nacheinander die zuvor gesammelten Videos analysiert. 

Für jedes Video berechnet `TransNetV2` pro Frame die Wahrscheinlichkeit für einen Einstellungsübergang. Dabei entstehen drei Ergebnisse:

- `video_frames` ist eine Liste der extrahierten Frames, die etwa zur Erstellung einer Visualisierung des Ergebnisses genutzt werden können.

- `all_frame_predictions` enthält die unbearbeiteten Wahrscheinlichkeiten für Einstellungsübergänge pro Frame.

- Für `single_frame_predictions` wurden im Falle einer Häufung hoher Wahrscheinlichkeiten in unmittelbarer Nähe eine einzelne, höchste Wahrscheinlichkeit ausgewählt. Dies verhindert, dass während schrittweisen Übergängen (wie Überblendungen, Auf- oder Abblenden) mehrere Einstellungsübergänge erkannt werden. Die folgenden Schritte nutzen daher `single_frame_predictions` anstelle von `all_frame_predictions`. 

Die Funktion `predictions_to_scenes` übersetzt die Wahrscheinlichkeiten anschließend mithilfe des gewählten Schwellenwerts `THRESHOLD` in konkrete Einstellungssegmente (`scenes`). Jedes Segment wird als Paar der Framepositionen Startframe und Endframe gespeichert. Mit "scenes" sind hier also eigentlich "shots" bzw. "Einstellungen" gemeint, die Benennung der Funktion mit `predictions_to_scenes` beruht auf deren Bezeichnung in der verwendeten Python Programmbibliothek. Der Schwellenwert `THRESHOLD` wird im Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) eingestellt.

Zusätzlich misst das Notebook die benötigte Rechenzeit für die Durchführung der Analyse (`elapsed_time_hh_mm_ss`).

In [7]:
# Destination directory for later export files.
export_directory = Path(EXPORT_PATH)

# Analyze each previously registered video in sequence.
for video_name, video in videos.items():
   # Start a timer to measure the analysis duration.
   start_time = time.perf_counter()

   # Calculate predictions without calculating gradients.
   # Gradients are only required when training models.
   # This reduces memory usage and speeds up computation.
   with torch.no_grad():
      video_frames, single_frame_predictions, all_frame_predictions = model.predict_video(video["path"])

      # Convert the PyTorch result to a NumPy array for further processing.
      prediction_scores = single_frame_predictions.cpu().detach().numpy()

      # Convert the probability scores into concrete shot segments.
      scenes = model.predictions_to_scenes(prediction_scores, THRESHOLD)

      # Calculate the elapsed time from when the timer was started.
      elapsed_time_seconds = time.perf_counter() - start_time
      # Convert the elapsed time into the hhmmss format
      elapsed_time_hh_mm_ss = str(
         datetime.timedelta(seconds=round(elapsed_time_seconds))
         )

      print(f"Shot Boundary Detection on {video_name} took {elapsed_time_hh_mm_ss} of processing time.")
      print(DIVIDER)

      # Add the analysis result to the video dictionary
      video.update({
         "scenes" : scenes,
         "elapsed_time_hh_mm_ss" : elapsed_time_hh_mm_ss,
         "prediction_scores": prediction_scores
      })

Extracting frames from videos/Der_geteilte_Himmel_DEFA-Trailer/Der geteilte Himmel - DEFA-Trailer.mp4


Processing frames: 100%|██████████| 5225/5225 [01:00<00:00, 85.94frame/s]

Shot Boundary Detection on Der geteilte Himmel - DEFA-Trailer took 0:01:06 of processing time.
────────────────────────────────────────────────────────────────────────────────────────────────────


### Tabellarische Aufbereitung des Ergebnisses

Die vom Modell gelieferten Einstellungen (`scenes`) werden in einen `pandas`-DataFrame (`df_shots`) überführt, mittels dessen sich Datensätze einfacher bearbeiten und exportieren lassen. Ein Dataframe ist dabei ein spezielles Tabellenformat. Jede Tabellenzeile entspricht einer erkannten Einstellung.

Neben Startframe (`start`) und Endframe (`end`) werden der zeitliche Mittelpunkt (`middle`) und die Dauer (`duration`) der Einstellung berechnet. Anschließend werden alle Frameangaben mithilfe der Bildrate (`frame_rate`) in Sekunden umgerechnet. Die Sekundenwerte erleichtern die weitere Auswertung der erkannten Einstellungen.


In [8]:
# Process the analysis results for each previously registered video in sequence.
for video_name, video in videos.items():
    # Build dataframe. Each row contains the start and end frame of a shot.
    df_shots = pd.DataFrame(video["scenes"], columns=["start", "end"])

    # Calculate the middle frame of a shot.
    df_shots["middle"] = df_shots["start"] + ((df_shots["end"] - df_shots["start"]) / 2).astype(int)

    # Calculate the duration of the shot in number of frames
    df_shots["duration"] = df_shots["end"] - df_shots["start"] + 1

    # Convert frames to seconds.
    # Time in seconds = frame number divided by frames per second ("frame_rate") of the respective video.
    for col in ["start", "end", "middle", "duration"]:
        df_shots[f"{col}_s"] = round(df_shots[col] / video["frame_rate"], 2)

    # Preview the first five detected shots.
    print(f"First five shots in {video_name}:")
    print(df_shots.head(5))
    print(DIVIDER)

    # Add the complete table to the corresponding video dictionary
    video.update({
        "df_shots" : df_shots
    })

First five shots in Der geteilte Himmel - DEFA-Trailer:
   start   end  middle  duration  start_s  end_s  middle_s  duration_s
0      0   179      89       180     0.00   7.16      3.56        7.20
1    180   447     313       268     7.20  17.88     12.52       10.72
2    448   788     618       341    17.92  31.52     24.72       13.64
3    789   971     880       183    31.56  38.84     35.20        7.32
4    972  1186    1079       215    38.88  47.44     43.16        8.60
────────────────────────────────────────────────────────────────────────────────────────────────────


### Berechnung der Analysemetadaten

Neben den Ergebnissen der Einstellungserkennung erzeugt das Notebook pro Video eine weitere Tabelle, in der die Metadaten zur Analyse eingetragen werden: Der angewendete Schwellenwert, die Anzahl der Einstellungen, der Mittelwert der Einstellungslängen, der Median der Einstellungslängen, die Dauer der längsten und der kürzesten Einstellung, die Gesamtdauer des Videos, die für die Analyse benötigte Rechenzeit, der minimale und der maximale Wahrscheinlichkeitswert bei der Erkennung, die Auflösung und die Bildrate der Videodatei. Diese Daten sind nützlich, um eine kompakte Übersicht über das Ergebnis zu erhalten oder um etwa die Auswirkungen unterschiedlicher `THRESHOLD`-Werte auf die Erkennungsergebnisse zu vergleichen.

Mittelwert (`mean`) und Median (`median`) sind dabei unterschiedliche Methoden zur Berechnung von Durchschnittswerten. Der Mittelwert wird berechnet indem die Gesamtdauer des Videos durch die Anzahl der Einstellungen in einem Video geteilt wird und reagiert stärker auf sehr lange Einstellungen. Der Median beschreibt hingegen die Dauer der Einstellung die genau in der Mitte liegt, wenn alle Einstellungen ihrer Länge nach nacheinander angeordnet werden (von der längsten bis zur kürzesten) und ist gegenüber Ausreißern robuster.


In [9]:
# Calculate the metadata for each previously registered video in sequence.
for video_name, video in videos.items():
    metadata = pd.Series({
        # Record the threshold used to ensure reproducibility.
        "threshold"                 : THRESHOLD,

        # Number of shots detected by the model.
        "shot_count"                : len(video["df_shots"]),

        # Descriptive statistics for shot duration in seconds.
        "mean_duration_s"           : round(video["df_shots"]["duration_s"].mean(), 2),
        "median_duration_s"         : round(video["df_shots"]["duration_s"].median(), 2),
        "max_duration_s"            : round(video["df_shots"]["duration_s"].max(), 2),
        "min_duration_s"            : round(video["df_shots"]["duration_s"].min(), 2),

        # Duration of the source video and time required for automatic processing.
        "duration_hh_mm_ss"          : video["duration_hh_mm_ss"],
        "processing_time_hh_mm_ss"  : video["elapsed_time_hh_mm_ss"],

        # Range of probability values produced by the model.
        "min_prediction_score"      : round(float(video["prediction_scores"].min()),2),
        "max_prediction_score"      : round(float(video["prediction_scores"].max()),2),

        # Video metadata
        "frame_width"               : video["frame_width"],
        "frame_height"              : video["frame_height"],
        "frame_rate"                       : video["frame_rate"]              
    })

    print(f"Metadata for {video_name}:")
    print(metadata)
    print(DIVIDER)

    # Store the metadata within the corresponding video dictionary.
    video.update({
        "metadata" : metadata
    })

Metadata for Der geteilte Himmel - DEFA-Trailer:
threshold                       0.5
shot_count                       23
mean_duration_s                9.09
median_duration_s              7.76
max_duration_s                22.04
min_duration_s                 1.96
duration_hh_mm_ss           0:03:29
processing_time_hh_mm_ss    0:01:06
min_prediction_score            0.0
max_prediction_score           0.99
frame_width                    1920
frame_height                   1080
frame_rate                     25.0
dtype: object
────────────────────────────────────────────────────────────────────────────────────────────────────


---
## Export der Ergebnisse

Die Analyseergebnisse werden als nächstes in mehreren Formaten gespeichert. `.csv`-Dateien eignen sich für Tabellenkalkulation, Statistikprogramme und weitere Python-Auswertungen, etwa zur Visualisierung. Zudem können sie auch  in *OpenRefine* eingelesen werden. Die Verwendung von *OpenRefine* wird in der OER zu <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-2/bereinigung/openRefine/1_einf%C3%BChrung.html" class="external-link" target="_blank">OER zu Studentischen Filmen an der Filmuniversität</a> genauer erklärt, ebenso das <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-2/bereinigung/openRefine/2_import.html#datenformate" class="external-link" target="_blank">csv-Dateiformat</a>.

Das `.eaf`-Format kann direkt in *VIAN* oder anderen Programmen zur Videoannotation geöffnet oder importiert werden, die Daten können dann dort manuell überprüft, bearbeitet und ergänzt werden.

Die Dateinamen enthalten am Ende den bei der Erkennung verwendeten Schwellenwert, z.B. `_0.5.csv` wie in `Der geteilte Himmel - DEFA-Trailer_shots_0.5.csv` oder `Der geteilte Himmel - DEFA-Trailer_metadata_0.5.csv`.

**[Question: should the video folder structure be reproduced during export?]**

### Export der Tabellen als `.csv`-Datei

In [10]:
# Export the tabular results for each previously registered video in sequence.
for video_name, video in videos.items():

  # Create the path and filename for the table of shots.
  # Including the threshold in the name makes different analysis runs distinguishable.
  shots_csv_path = export_directory / f"{video_name}_shots_{THRESHOLD}.csv"

  # Intentionally replace any existing file with the same name.
  if shots_csv_path.is_file():
    shots_csv_path.unlink()
    print(f"Overwriting {shots_csv_path}.")

  # Write the .csv file for the table of shots.
  # "index=False" prevents an additional row-number column that is not needed for the data.
  video["df_shots"].to_csv(shots_csv_path, index=False)

  # Save a separate .csv file for the metadata table
  metadata_csv_path = export_directory / f"{video_name}_metadata_{THRESHOLD}.csv"
  if metadata_csv_path.is_file():
    metadata_csv_path.unlink()
    print(f"Overwriting {metadata_csv_path}.")
  # Saving without a header produces simple key-value pairs.
  video["metadata"].to_csv(metadata_csv_path, header=False)

  print(f"Successfully exported {shots_csv_path} & {metadata_csv_path}.")
  print(DIVIDER)

Successfully exported export/Der geteilte Himmel - DEFA-Trailer_shots_0.5.csv & export/Der geteilte Himmel - DEFA-Trailer_metadata_0.5.csv.
────────────────────────────────────────────────────────────────────────────────────────────────────


### Export der Einstellungsannotationen als `.eaf`-Datei

Das `.eaf`-Format speichert Annotationen als Zeitangaben in Millisekunden. Deshalb werden die Start- und Endframes jeder Einstellung mithilfe der individuellen Bildrate des Videos direkt in Millisekunden umgerechnet. Der Endzeitpunkt liegt unmittelbar nach dem letzten zur Einstellung gehörenden Frame.

Für jedes Video wird eine Annotationsebene namens `shots` angelegt. Jede erkannte Einstellung erscheint dort als eigenes Zeitintervall. Die Intervalle enthalten anders als gewöhnlich keinen Text, sondern bilden eine zeitliche Segmentierung, die nach dem Import in VIAN dort auf ihre korrekte Erkennung hin überprüft und inhaltlich ergänzt werden kann. **[MAYBE CHANGE THIS BEHAVIOUR]**

In [11]:
# Export the .eaf annotations for each previously registered video in sequence.
for video_name, video in videos.items():
  
    # Create a new, initially empty .eaf document.
    eaf_document = pympi.Eaf()
    # Add the annotation tier for the automatically detected shots.
    eaf_document.add_tier("shots")

     # Convert each table row into a time interval in milliseconds.
    for _, row in video["df_shots"].iterrows():
        eaf_document.add_annotation(
            "shots",
            round(row["start"] / video["frame_rate"] * 1000),
            round((row["end"] + 1) / video["frame_rate"] * 1000)
        )

    # Create the path and filename for the .eaf file.
    eaf_document_path = export_directory / f"{video_name}_shots_{THRESHOLD}.eaf"

    # Intentionally replace any existing file with the same name.
    if eaf_document_path.is_file():
        eaf_document_path.unlink()
        print(f"Overwriting {eaf_document_path}.")

    # Write the EAF file to disk.
    eaf_document.to_file(eaf_document_path)

    print(f"Successfully exported {eaf_document_path}.")
    print(DIVIDER)

Successfully exported export/Der geteilte Himmel - DEFA-Trailer_shots_0.5.eaf.
────────────────────────────────────────────────────────────────────────────────────────────────────


### Herunterladen des *Google Colab* Export-Ordners

Dieser optionale Schritt ist nur für eine Ausführung in *Google Colab* vorgesehen. Der gesamte Exportordner in *Colab* wird zu einer `.zip`-Datei komprimiert und anschließend über die Browseroberfläche heruntergeladen. Der Download ist notwendig, da ansonsten die Erkennungsdaten verloren gehen, da diese nach Beenden der Laufzeit bzw. von *Colab* immer gelöscht werden, ebenso wie alle in Colab zur Analyse hochgeladenen Videodateien.

In einer lokalen *Jupyter Notebook*-Umgebung bleiben die Dateien im anfangs im Codeblock zur [Festlegung der Parameter](#1-festlegen-der-parameter) definierten Export-Ordner erhalten und können von dort entnommen werden. Es ist daher kein zusätzlicher Download-Schritt erforderlich.


In [12]:
try:
  # The following import works only within a Google Colab runtime.
  import google.colab
  from google.colab import files

  # Colab uses the `/content` directory by default.
  colab_export_directory = f"/content/{EXPORT_PATH}"
  zip_path = f"{EXPORT_PATH}.zip"
  # Package the entire results directory as a ZIP archive.
  !zip -r "{zip_path}" "{colab_export_directory}"

  # Start downloading the ZIP archive in the browser.
  files.download(f"/content/{EXPORT_PATH}.zip")
  print(f"Downloaded {EXPORT_PATH}.zip.")
except ImportError:
  # Skip this step when running locally.
  print("Download only works in Google Colab.")

Download only works in Google Colab.
